# EDA

先看看数据长什么样，搞清楚趋势、季节性这些基本的东西。

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import STL
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller

from src.data_loader import load_raw_data, get_monthly_sales, get_category_monthly

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
%matplotlib inline

In [ ]:
df = load_raw_data('../data/raw/train.csv')
print(f'{len(df)} rows')
df.head()

In [ ]:
df.info()

In [ ]:
print(f"日期: {df['Order Date'].min()} ~ {df['Order Date'].max()}")
print(f"品类: {df['Category'].unique()}")
print(f"区域: {df['Region'].unique()}")
print(f"子品类数: {df['Sub-Category'].nunique()}")

4年数据，聚合到月度的话只有48个点，后面LSTM估计要吃亏。

## 月度趋势

In [ ]:
monthly = get_monthly_sales(df)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly.index, monthly.values, marker='o', markersize=3, color='steelblue')
ax.set_title('Monthly Total Sales')
ax.set_ylabel('Sales ($)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../results/figures/monthly_trend.png', dpi=150, bbox_inches='tight')
plt.show()

趋势在涨，每年年底有个peak（大概是Black Friday和圣诞），这种季节性应该比较好建模。

## 按品类看

In [ ]:
cat_monthly = get_category_monthly(df)

fig, ax = plt.subplots(figsize=(14, 5))
for cat in cat_monthly['category'].unique():
    mask = cat_monthly['category'] == cat
    ax.plot(cat_monthly.loc[mask, 'date'], cat_monthly.loc[mask, 'sales'], 
            label=cat, marker='o', markersize=2)
ax.legend()
ax.set_title('Monthly Sales by Category')
ax.set_ylabel('Sales ($)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../results/figures/category_trend.png', dpi=150, bbox_inches='tight')
plt.show()

Technology波动最大。先用总量建模吧，分品类建模以后再说。

## 区域 & 品类分布

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

df.groupby('Region')['Sales'].sum().sort_values(ascending=True).plot(
    kind='barh', ax=ax1, color='steelblue')
ax1.set_title('Total Sales by Region')

df.groupby('Category')['Sales'].sum().sort_values(ascending=True).plot(
    kind='barh', ax=ax2, color='coral')
ax2.set_title('Total Sales by Category')

plt.tight_layout()
plt.savefig('../results/figures/region_category_bar.png', dpi=150, bbox_inches='tight')
plt.show()

## STL分解

In [ ]:
stl = STL(monthly, period=12)
res = stl.fit()

fig = res.plot()
fig.set_size_inches(12, 8)
plt.tight_layout()
plt.savefig('../results/figures/stl_decomposition.png', dpi=150, bbox_inches='tight')
plt.show()

trend稳步上升，seasonal很规律（振幅大概±15k），residual没什么明显pattern。

季节性这么强的话，Seasonal Naive说不定就挺好使。

## ACF / PACF

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(monthly, lags=24, ax=ax1)
plot_pacf(monthly, lags=24, ax=ax2, method='ywm')
plt.tight_layout()
plt.savefig('../results/figures/acf_pacf.png', dpi=150, bbox_inches='tight')
plt.show()

ACF缓慢衰减 → 需要差分；lag12有明显spike → 年度季节性。PACF lag1截尾，AR(1)。

## ADF检验

In [ ]:
def run_adf(series, name=''):
    stat, pval, _, _, _, _ = adfuller(series, autolag='AIC')
    print(f'{name:25s}  ADF={stat:.4f}  p={pval:.4f}  {"✓ stationary" if pval < 0.05 else "✗ non-stationary"}')

run_adf(monthly, 'Original')
run_adf(monthly.diff().dropna(), '1st diff')
run_adf(monthly.diff(12).dropna(), 'Seasonal diff (12)')

一阶差分就够了，d=1。

## 月份分布

In [ ]:
mdf = pd.DataFrame({'sales': monthly, 'month': monthly.index.month})

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=mdf, x='month', y='sales', ax=ax, color='lightblue')
ax.set_title('Sales by Month')
plt.tight_layout()
plt.savefig('../results/figures/monthly_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

9月和11-12月最高，1-2月低谷。上升趋势+强季节性(m=12)，d=1平稳。